# Ensemble BO for f1–f3

Updated strategy:
- ARD GP ensemble (Matern + RBF)
- SVC gating in PCA space
- Dimension-scaled distance in candidate generation / diversity bonus
- **No neural-network surrogate for this group** (sample sizes remain too small)


In [1]:
import numpy as np
import pandas as pd
import math
from pathlib import Path

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, RBF, WhiteKernel, ConstantKernel
from sklearn.svm import LinearSVC

import warnings
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

# --- Config ---
DATA_PATH = Path("data.csv")
FUNCTIONS = ['function_1', 'function_2', 'function_3']
DIMS = {'function_1': 2, 'function_2': 2, 'function_3': 3}
CURRENT_STEP = 9
N_GLOBAL = 8000
N_LOCAL = 3000
N_PCA = 3000
LOW, HIGH = 0.0, 1.0
RANDOM_SEED = 19
IS_MAXIMISATION = True
DEBUG = True
SVC_GATE_KEEP_FRAC = 0.5
SVC_LABEL_QUANTILE = 0.75
DISTANCE_BONUS = 0.05
LOCAL_RADIUS = 0.15


def normal_pdf(z):
    return (1.0 / np.sqrt(2.0 * np.pi)) * np.exp(-0.5 * z**2)


def normal_cdf(z):
    return 0.5 * (1.0 + np.vectorize(math.erf)(z / np.sqrt(2.0)))


def acquisition_ucb(mu, sigma, kappa):
    return mu + kappa * sigma


def acquisition_ei(mu, sigma, best_y, xi):
    sigma = np.maximum(sigma, 1e-12)
    imp = mu - best_y - xi
    z = imp / sigma
    return imp * normal_cdf(z) + sigma * normal_pdf(z)


def read_data(path: Path):
    if not path.exists():
        raise FileNotFoundError(f"Could not find {path}.")
    df = pd.read_csv(path)
    df["function"] = df["function"].astype(str).str.strip()
    return df


def get_xy(df: pd.DataFrame, fname: str, d: int):
    sub = df[df["function"] == fname].copy()
    if sub.empty:
        raise ValueError(f"No rows found for function={fname} in {DATA_PATH}.")
    X = sub[[f"x{i}" for i in range(1, d + 1)]].to_numpy(dtype=float)
    y = sub["y"].to_numpy(dtype=float).reshape(-1)
    m = np.isfinite(X).all(axis=1) & np.isfinite(y)
    return X[m], y[m]


def format_output_line(fname, x):
    parts = [fname] + [f"{v:.6f}" for v in x.tolist()]
    return ", ".join(parts)


def minmax01(a):
    a = np.asarray(a, dtype=float)
    lo, hi = np.min(a), np.max(a)
    if hi - lo < 1e-12:
        return np.zeros_like(a)
    return (a - lo) / (hi - lo)


def safe_normalise(w, floor=1e-3):
    w = np.asarray(w, dtype=float)
    w = np.where(np.isfinite(w), w, 0.0)
    w = np.maximum(w, 0.0)
    if w.sum() <= 0:
        w = np.ones_like(w)
    w = w / w.sum()
    w = np.maximum(w, floor)
    return w / w.sum()


def weights_to_scales(w, clip=(0.35, 2.5)):
    w = safe_normalise(w)
    scales = np.sqrt(w * w.size)
    return np.clip(scales, clip[0], clip[1])


def scaled_distance(X, x0, w):
    X = np.asarray(X, dtype=float)
    x0 = np.asarray(x0, dtype=float)
    w = safe_normalise(w)
    diff = X - x0.reshape(1, -1)
    return np.sqrt(np.sum(w.reshape(1, -1) * diff * diff, axis=1))


def min_scaled_distance_to_observed(C, X, w):
    w = safe_normalise(w)
    diff = C[:, None, :] - X[None, :, :]
    d2 = np.sum(w.reshape(1, 1, -1) * diff * diff, axis=2)
    return np.sqrt(np.min(d2, axis=1))


def sample_uniform(n, d, rng):
    return rng.uniform(LOW, HIGH, size=(n, d))


def sample_local_weighted(best_x, n, rng, radius, dim_weights):
    scales = weights_to_scales(dim_weights)
    X = best_x.reshape(1, -1) + rng.normal(0.0, radius, size=(n, best_x.size)) * scales.reshape(1, -1)
    return np.clip(X, LOW, HIGH)


def pca_guided_candidates(X, n, rng, n_components=2, expand=0.75):
    d = X.shape[1]
    n_components = min(n_components, d)
    scaler = StandardScaler()
    Xs = scaler.fit_transform(X)
    pca = PCA(n_components=n_components, random_state=0)
    Z = pca.fit_transform(Xs)
    z_lo = Z.min(axis=0)
    z_hi = Z.max(axis=0)
    span = np.where((z_hi - z_lo) == 0, 1.0, (z_hi - z_lo))
    Zc = rng.uniform(z_lo - expand * span, z_hi + expand * span, size=(n, n_components))
    Xc = scaler.inverse_transform(pca.inverse_transform(Zc))
    return np.clip(Xc, LOW, HIGH)


def schedule_params(step_idx):
    if step_idx <= 4:
        return "ucb", 5.0, None
    if step_idx <= 9:
        return "ei", None, 0.03
    return "ei", None, 0.005


def fit_gp_models(X, y, seed=0):
    scaler = StandardScaler()
    Xs = scaler.fit_transform(X)
    d = X.shape[1]
    kernels = [
        ConstantKernel(1.0, (1e-3, 1e3)) * Matern(length_scale=np.ones(d), length_scale_bounds=(1e-3, 1e3), nu=2.5)
        + WhiteKernel(noise_level=1e-6, noise_level_bounds=(1e-8, 1e-1)),
        ConstantKernel(1.0, (1e-3, 1e3)) * RBF(length_scale=np.ones(d), length_scale_bounds=(1e-3, 1e3))
        + WhiteKernel(noise_level=1e-6, noise_level_bounds=(1e-8, 1e-1)),
    ]
    gps = []
    for k in kernels:
        gp = GaussianProcessRegressor(
            kernel=k,
            normalize_y=True,
            n_restarts_optimizer=4,
            random_state=seed,
        )
        gp.fit(Xs, y)
        gps.append(gp)
    return gps, scaler


def extract_length_scales(kernel):
    if hasattr(kernel, "k1") and hasattr(kernel.k1, "k2") and hasattr(kernel.k1.k2, "length_scale"):
        ls = np.asarray(kernel.k1.k2.length_scale, dtype=float)
        return ls.reshape(-1)
    if hasattr(kernel, "length_scale"):
        return np.asarray(kernel.length_scale, dtype=float).reshape(-1)
    for attr in ["k1", "k2"]:
        if hasattr(kernel, attr):
            out = extract_length_scales(getattr(kernel, attr))
            if out is not None:
                return out
    return None


def ard_weights_from_gp_ensemble(gps, d):
    all_w = []
    for gp in gps:
        ls = extract_length_scales(gp.kernel_)
        if ls is None:
            continue
        if ls.size == 1:
            ls = np.repeat(ls, d)
        rel = 1.0 / np.maximum(ls, 1e-9)
        all_w.append(safe_normalise(rel))
    if not all_w:
        return np.ones(d) / d
    return safe_normalise(np.mean(np.vstack(all_w), axis=0))


def ensemble_predict(gps, scaler, Xcand):
    Xs = scaler.transform(Xcand)
    mus = []
    vars_ = []
    for gp in gps:
        mu, std = gp.predict(Xs, return_std=True)
        mus.append(mu)
        vars_.append(std ** 2)
    mus = np.vstack(mus)
    vars_ = np.vstack(vars_)
    mu_ens = np.mean(mus, axis=0)
    var_ens = np.mean(vars_, axis=0) + np.var(mus, axis=0)
    return mu_ens, np.sqrt(np.maximum(var_ens, 1e-12))


def make_svc_labels(y, quantile=0.75, maximise=True):
    thr = np.quantile(y, quantile if maximise else 1 - quantile)
    labels = (y >= thr).astype(int) if maximise else (y <= thr).astype(int)
    return labels, thr


def svc_gate_filter_candidates_pca(X, y, C, n_components=2, keep_frac=0.5, quantile=0.75, maximise=True):
    scaler = StandardScaler()
    Z = scaler.fit_transform(X)
    pca = PCA(n_components=min(n_components, X.shape[1]), random_state=0)
    Z = pca.fit_transform(Z)
    labels, thr = make_svc_labels(y, quantile=quantile, maximise=maximise)
    if np.unique(labels).size < 2:
        if DEBUG:
            print(f"[SVC gate | PCA] skipped; single class at y_thr={thr:.4g}")
        return C, None
    clf = LinearSVC(C=1.0, class_weight="balanced", dual=(Z.shape[0] <= Z.shape[1]))
    clf.fit(Z, labels)
    Zc = pca.transform(scaler.transform(C))
    margin = clf.decision_function(Zc)
    k = max(1, int(np.ceil(keep_frac * len(C))))
    idx = np.argsort(margin)[-k:]
    C_f = C[idx]
    if DEBUG:
        print(f"[SVC gate | PCA{Z.shape[1]}] y_thr={thr:.4g} kept={len(C_f)}/{len(C)}")
    return C_f, margin[idx]


def propose_next(fname, X, y, rng):
    mode, kappa, xi = schedule_params(CURRENT_STEP)
    best_idx = int(np.argmax(y))
    best_x = X[best_idx].copy()
    best_y = float(y[best_idx])

    gps, scaler = fit_gp_models(X, y, seed=RANDOM_SEED)
    dim_weights = ard_weights_from_gp_ensemble(gps, X.shape[1])

    C = [
        sample_uniform(N_GLOBAL, X.shape[1], rng),
        sample_local_weighted(best_x, N_LOCAL, rng, radius=LOCAL_RADIUS, dim_weights=dim_weights),
        pca_guided_candidates(X, N_PCA // 2, rng, n_components=1),
    ]
    if X.shape[1] >= 2:
        C.append(pca_guided_candidates(X, N_PCA // 2, rng, n_components=2))
    C = np.vstack(C)
    C, _ = svc_gate_filter_candidates_pca(
        X, y, C,
        n_components=min(2, X.shape[1]),
        keep_frac=SVC_GATE_KEEP_FRAC,
        quantile=SVC_LABEL_QUANTILE,
        maximise=IS_MAXIMISATION,
    )

    mu, std = ensemble_predict(gps, scaler, C)
    score = acquisition_ucb(mu, std, kappa=kappa) if mode == "ucb" else acquisition_ei(mu, std, best_y=best_y, xi=xi)
    score = score + DISTANCE_BONUS * minmax01(min_scaled_distance_to_observed(C, X, dim_weights))

    if DEBUG:
        print(f"[{fname}] ARD weights = {np.round(dim_weights, 3)}")
    return C[int(np.argmax(score))]


df = read_data(DATA_PATH)
rng = np.random.default_rng(RANDOM_SEED)
for fname in FUNCTIONS:
    d = DIMS[fname]
    X, y = get_xy(df, fname, d)
    x_next = propose_next(fname, X, y, rng)
    print(format_output_line(fname, x_next))


[SVC gate | PCA2] y_thr=7.904e-47 kept=7000/14000
[function_1] ARD weights = [0.999 0.001]
function_1, 0.942157, 0.923095
[SVC gate | PCA2] y_thr=0.5094 kept=7000/14000
[function_2] ARD weights = [0.999 0.001]
function_2, 0.514542, 1.000000
[SVC gate | PCA2] y_thr=-0.0623 kept=7000/14000
[function_3] ARD weights = [0.001 0.053 0.946]
function_3, 0.765413, 0.588176, 0.495840
